In [3]:
import pandas as pd
import hashlib
from datetime import date
import numpy as np

# --- 1. Load the Master Dataset and Simulate Daily Snapshots ---

try:
    master_df_path = '/content/drive/MyDrive/MCA/MCA_Project_Data/mca_master_dataset.csv'
    day1_df = pd.read_csv(master_df_path, low_memory=False)
    print("Successfully loaded 'mca_master_dataset.csv' as Day 1.")
except FileNotFoundError:
    print(f"File not found at {master_df_path}. Please check the path.")
    exit()

# --- 2. Standardize Columns and Handle Duplicates ---
cin_col = 'corporateidentificationnumber'

day1_df.columns = day1_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('_', '')

# Robustly find and rename the primary CIN column
for col_name in day1_df.columns:
    if 'corporate' in col_name or 'cin' in col_name:
        day1_df.rename(columns={col_name: cin_col}, inplace=True)
        break # Stop after renaming the first match

day1_df = day1_df.loc[:, ~day1_df.columns.duplicated()]
# -------------------------------------------------------

# --- Create Day 2 Data by Simulating Changes ---
day2_df = day1_df.copy()

# A. Simulate Status Changes
change_indices = day2_df.sample(n=10, random_state=1).index
day2_df.loc[change_indices, 'companystatus'] = 'Strike Off'

# B. Simulate Capital Modification
capital_indices = day2_df.drop(change_indices).sample(n=10, random_state=2).index
day2_df.loc[capital_indices, 'authorizedcapital'] = day2_df.loc[capital_indices, 'authorizedcapital'] * 1.25

# C. Simulate New Incorporations
new_companies = [
    {'corporateidentificationnumber': 'N00001MH2025PTC00001', 'companyname': 'NEW GEN AI SOLUTIONS LTD', 'companystatus': 'Active', 'authorizedcapital': 500000, 'paidupcapital': 100000, 'companyclass': 'Private'},
    {'corporateidentificationnumber': 'N00002DL2025PLC00002', 'companyname': 'QUANTUM LEAP LOGISTICS PLC', 'companystatus': 'Active', 'authorizedcapital': 2000000, 'paidupcapital': 800000, 'companyclass': 'Public'},
    {'corporateidentificationnumber': 'N00003KA2025PTC00003', 'companyname': 'BIOVITA HEALTHCARE PRIVATE LIMITED', 'companystatus': 'Active', 'authorizedcapital': 300000, 'paidupcapital': 75000, 'companyclass': 'Private'},
]
day2_df = pd.concat([day2_df, pd.DataFrame(new_companies)], ignore_index=True)
print("Successfully created simulated Day 2 data with changes.")

# --- 3. Generate Hashes ---
def create_row_digest(df, columns_to_hash):
    existing_cols = [col for col in columns_to_hash if col in df.columns]
    if not existing_cols: return None
    combined_string = df[existing_cols].astype(str).agg(''.join, axis=1)
    return combined_string.apply(lambda x: hashlib.md5(x.encode()).hexdigest())

cols_to_track = ['companystatus', 'authorizedcapital', 'paidupcapital', 'companyclass']
day1_df['row_digest'] = create_row_digest(day1_df, cols_to_track)
day2_df['row_digest'] = create_row_digest(day2_df, cols_to_track)


# --- 4. Compare Digests and Find Changes ---
merged_df = day1_df.merge(
    day2_df,
    on=cin_col,
    how='outer',
    suffixes=('_old', '_new')
)

name_col_old = 'companyname_old'
name_col_new = 'companyname_new'
changes = []
today_str = date.today().strftime('%Y-%m-%d')

# Find New Incorporations
new_registrations = merged_df[merged_df['row_digest_old'].isnull()]
for index, row in new_registrations.iterrows():
    changes.append({
        'CIN': row[cin_col], 'Change_Type': 'New Registration', 'Field_Changed': 'N/A',
        'Old_Value': 'N/A', 'New_Value': row.get(name_col_new, 'N/A'), 'Date': today_str
    })

# Find Field Updates
updates = merged_df.dropna(subset=['row_digest_old', 'row_digest_new'])
updates = updates[updates['row_digest_old'] != updates['row_digest_new']]

for index, row in updates.iterrows():
    for col in cols_to_track:
        old_val = row.get(f'{col}_old')
        new_val = row.get(f'{col}_new')
        if pd.notna(old_val) and pd.notna(new_val) and str(old_val) != str(new_val):
            changes.append({
                'CIN': row[cin_col], 'Change_Type': 'Field Update', 'Field_Changed': col,
                'Old_Value': old_val, 'New_Value': new_val, 'Date': today_str
            })


# --- 5. Generate and Save the Final Change Log ---
change_log_df = pd.DataFrame(changes)

print(f"\nChange Detection Complete:")
print(f" - New Registrations: {len(new_registrations)}")
print(f" - Field Updates Found: {len(updates)}")

print("\nSample of Change Log:")
print(change_log_df.head())

# Save the log to a CSV file in your Drive
output_path = '/content/drive/MyDrive/MCA/change_log.csv'
change_log_df.to_csv(output_path, index=False)
print(f"\nSuccessfully saved the results to: {output_path}")

Successfully loaded 'mca_master_dataset.csv' as Day 1.
Successfully created simulated Day 2 data with changes.

Change Detection Complete:
 - New Registrations: 3
 - Field Updates Found: 25

Sample of Change Log:
                     CIN       Change_Type      Field_Changed  Old_Value  \
0   N00001MH2025PTC00001  New Registration                N/A        N/A   
1   N00002DL2025PLC00002  New Registration                N/A        N/A   
2   N00003KA2025PTC00003  New Registration                N/A        N/A   
3  U15149GJ2003PTC042265      Field Update  authorizedcapital  3200000.0   
4  U15149GJ2003PTC042265      Field Update  authorizedcapital  3200000.0   

                            New_Value        Date  
0            NEW GEN AI SOLUTIONS LTD  2025-10-19  
1          QUANTUM LEAP LOGISTICS PLC  2025-10-19  
2  BIOVITA HEALTHCARE PRIVATE LIMITED  2025-10-19  
3                           4000000.0  2025-10-19  
4                           4000000.0  2025-10-19  

Successfully save

In [4]:
import pandas as pd
# Load the change log you created in Task B
try:
    change_log_df = pd.read_csv('/content/drive/MyDrive/MCA/change_log.csv')

    # Get a list of unique CINs
    unique_cins = change_log_df['CIN'].unique()
    num_unique_cins = len(unique_cins)

    # Determine the sample size: 50 or the total number of unique CINs if less than 50
    sample_size = min(50, num_unique_cins)

    # Take the sample
    sample_cins = pd.Series(unique_cins).sample(n=sample_size, random_state=42).tolist()

    print(f"Found {num_unique_cins} unique CINs.")
    print(f"Selected a sample of {len(sample_cins)} CINs for enrichment.")
    print("Sample CINs:", sample_cins[:5]) # Print the first 5

except FileNotFoundError:
    print("File not found! Please check the path to 'change_log.csv'.")

Found 19 unique CINs.
Selected a sample of 19 CINs for enrichment.
Sample CINs: ['N00001MH2025PTC00001', 'U24110MH1999PTC121310', 'U70109DL2012PTC232485', 'N00002DL2025PLC00002', 'U63030DL2022PTC405524']


In [4]:
!pip install Faker

In [5]:
import pandas as pd
from faker import Faker
import random

# Initialize the Faker library
fake = Faker('en_IN')

# This is the list of CINs from your previous step
# sample_cins = ['U85100KA2022PTC163803', 'U26915KA2000PLC026915', ...]

enriched_data = []

print("Starting simulated enrichment process for 50 companies...")

# Loop through each CIN in your sample list
for cin in sample_cins:
    # For each company, generate 2 to 4 fake director names
    num_directors = random.randint(2, 4)
    for _ in range(num_directors):
        director_name = fake.name()

        # Append the simulated data in the required format
        enriched_data.append({
            'CIN': cin,
            'COMPANY_NAME': fake.company() + ' ' + fake.company_suffix(),
            'STATE': fake.state(),
            'STATUS': 'Active',
            'SOURCE': 'Simulated Data',
            'FIELD': 'Director Name',
            'VALUE': director_name,
            'SOURCE_URL': 'N/A'
        })

print("\nEnrichment complete!")

# --- Convert the results into a DataFrame ---
enriched_df = pd.DataFrame(enriched_data)

print("\nSample of Enriched Data:")
print(enriched_df.head())

# Define the full path to your destination folder in Google Drive
output_path = '/content/drive/MyDrive/MCA/enriched_cin_data.csv'
# Save the DataFrame to the specified path
enriched_df.to_csv(output_path, index=False)
print(f"\nSuccessfully saved enriched data to: {output_path}")

Starting simulated enrichment process for 50 companies...

Enrichment complete!

Sample of Enriched Data:
                     CIN                      COMPANY_NAME              STATE  \
0   N00001MH2025PTC00001                Golla and Sons LLC          Karnataka   
1   N00001MH2025PTC00001                    Walia-Hora PLC     Andhra Pradesh   
2  U24110MH1999PTC121310  Arya, Ranganathan and Grover PLC              Bihar   
3  U24110MH1999PTC121310   Chaudhari, Kapur and Mangal PLC            Tripura   
4  U24110MH1999PTC121310  Singhal, Toor and Garde and Sons  Arunachal Pradesh   

   STATUS          SOURCE          FIELD            VALUE SOURCE_URL  
0  Active  Simulated Data  Director Name     Bhavani Chad        N/A  
1  Active  Simulated Data  Director Name     Nidra Thaman        N/A  
2  Active  Simulated Data  Director Name  Urishilla Bandi        N/A  
3  Active  Simulated Data  Director Name       Faris Bahl        N/A  
4  Active  Simulated Data  Director Name       Zilmi

In [13]:
!pip install streamlit -q

In [7]:
import pandas as pd
from datetime import date

# --- 1. Load the Change Log ---
try:
    # Use the full path to your change log file
    change_log_path = '/content/drive/MyDrive/MCA/change_log.csv'
    change_log_df = pd.read_csv(change_log_path)
    print("Successfully loaded change_log.csv")
except FileNotFoundError:
    print(f"File not found at {change_log_path}. Please check the path.")
    exit()

# --- 2. Generate the Summary ---
# Count the number of occurrences for each type of change
change_counts = change_log_df['Change_Type'].value_counts()

# Get the specific counts, with a default of 0 if a type doesn't exist
new_regs = change_counts.get('New Registration', 0)
updates = change_counts.get('Field Update', 0)
deregs = change_counts.get('Deregistration', 0)
total_changes = len(change_log_df)

# Get the current date for the report
report_date = date.today().strftime('%Y-%m-%d')

# --- 3. Create the Summary Text ---
summary_text = f"""
MCA Daily Insights Summary
===========================
Report Date: {report_date}

Key changes detected in the last cycle:

- New Company Registrations: {new_regs}
- Existing Company Updates: {updates}
- Company Deregistrations: {deregs}

---------------------------
Total Records Updated: {total_changes}
"""

print("\nGenerated Summary:")
print(summary_text)

# --- 4. Save the Summary to a File ---
# Define the output path in your Google Drive
output_path = '/content/drive/MyDrive/MCA/daily_summary.txt'

with open(output_path, 'w') as f:
    f.write(summary_text)

print(f"\n Successfully saved the summary to: {output_path}")

Successfully loaded change_log.csv

Generated Summary:

MCA Daily Insights Summary
Report Date: 2025-10-19

Key changes detected in the last cycle:

- New Company Registrations: 3
- Existing Company Updates: 25
- Company Deregistrations: 0

---------------------------
Total Records Updated: 28


 Successfully saved the summary to: /content/drive/MyDrive/MCA/daily_summary.txt


In [14]:
%%writefile app.py

import streamlit as st
import pandas as pd

# --- Page Configuration ---
st.set_page_config(
    page_title="MCA Insights Engine",
    page_icon="📊",
    layout="wide"
)

# --- Data Loading and Caching ---
@st.cache_data
def load_data():
    master_path = '/content/drive/MyDrive/MCA/MCA_Project_Data/mca_master_dataset.csv'
    change_log_path = '/content/drive/MyDrive/MCA/change_log.csv'
    enriched_path = '/content/drive/MyDrive/MCA/enriched_cin_data.csv'

    try:
        master_df = pd.read_csv(master_path, low_memory=False)
        change_log_df = pd.read_csv(change_log_path)
        enriched_df = pd.read_csv(enriched_path)

        master_df.columns = master_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('_', '')

        date_col = next((col for col in master_df.columns if 'date' in col and 'incorporation' in col), None)

        if date_col:
            master_df['incorporation_year'] = pd.to_datetime(master_df[date_col], errors='coerce').dt.year
        else:
            master_df['incorporation_year'] = pd.Series(dtype='float')

        return master_df, change_log_df, enriched_df

    except FileNotFoundError as e:
        st.error(f"Error loading data: {e}. Please check file paths.")
        return None, None, None

master_df, change_log_df, enriched_df = load_data()

# --- Main App ---
st.title("MCA Insights Engine")
st.markdown("An interactive dashboard to explore company data, track changes, and view enriched insights.")

if master_df is not None:
    # --- Robustly find all necessary column names ---
    cin_col = next((col for col in master_df.columns if 'corporate' in col or 'cin' in col), None)
    name_col = next((col for col in master_df.columns if 'companyname' in col), None)
    state_col = next((col for col in master_df.columns if 'state' in col), None)
    status_col = next((col for col in master_df.columns if 'status' in col), None)

    # --- Sidebar Filters ---
    st.sidebar.header("Filter Options")
    search_query = st.sidebar.text_input("Search by Company Name or CIN")

    valid_years = master_df['incorporation_year'].dropna()
    if not valid_years.empty:
        min_year, max_year = int(valid_years.min()), int(valid_years.max())
        selected_year = st.sidebar.slider("Filter by Year", min_year, max_year, (min_year, max_year))
    else:
        st.sidebar.warning("No valid year data.")
        selected_year = (2000, 2025)

    if state_col:
        states = master_df[state_col].dropna().unique()
        selected_states = st.sidebar.multiselect("Filter by State", states, default=list(states))
    if status_col:
        statuses = master_df[status_col].dropna().unique()
        selected_statuses = st.sidebar.multiselect("Filter by Company Status", statuses, default=list(statuses))

    # --- Filtering Logic ---
    filtered_df = master_df.copy()
    if not valid_years.empty:
        filtered_df = filtered_df[filtered_df['incorporation_year'].between(selected_year[0], selected_year[1])]
    if state_col:
        filtered_df = filtered_df[filtered_df[state_col].isin(selected_states)]
    if status_col:
        filtered_df = filtered_df[filtered_df[status_col].isin(selected_statuses)]
    if search_query and cin_col and name_col:
        filtered_df = filtered_df[
            filtered_df[name_col].str.contains(search_query, case=False, na=False) |
            filtered_df[cin_col].str.contains(search_query, case=False, na=False)
        ]

    # --- Display Data ---
    st.header("Company Data Explorer")
    st.dataframe(filtered_df.head(1000))
    st.markdown(f"Displaying **{min(len(filtered_df), 1000)}** of **{len(filtered_df)}** filtered companies.")

    # --- Display Detailed Insights ---
    st.header("Detailed Insights")
    if cin_col and not filtered_df.empty:
        selected_cin = st.selectbox("Select a CIN to see details", options=filtered_df[cin_col].unique())
        if selected_cin:
            col1, col2 = st.columns(2)
            with col1:
                st.subheader("Enriched Information")
                enriched_info = enriched_df[enriched_df['CIN'] == selected_cin]
                if not enriched_info.empty:
                    st.dataframe(enriched_info[['FIELD', 'VALUE', 'SOURCE']])
                else:
                    st.info("No enrichment data available.")
            with col2:
                st.subheader("Change History")
                change_history = change_log_df[change_log_df['CIN'] == selected_cin]
                if not change_history.empty:
                    st.dataframe(change_history[['Change_Type', 'Field_Changed', 'Old_Value', 'New_Value', 'Date']])
                else:
                    st.info("No change history available.")
            # -----------------------------------------------------------

    # --- E. AI Chatbot ---
    st.header("AI Chatbot")
    st.markdown("Ask questions about the latest changes.")

    def get_bot_response(question, log_df):
        """A simple rule-based chatbot function."""
        question = question.lower()

        if "new" in question and ("how many" in question or "count" in question):
            count = log_df[log_df['Change_Type'] == 'New Registration'].shape[0]
            return f"There were {count} new company registrations in the last cycle."

        elif "update" in question and ("how many" in question or "count" in question):
            count = log_df[log_df['Change_Type'] == 'Field Update'].shape[0]
            return f"There were {count} updates to existing companies."

        elif "total" in question and "change" in question:
            count = log_df.shape[0]
            return f"In total, there were {count} changes detected."

        else:
            return "Sorry, I can only answer questions about the count of new registrations or updates. Try 'How many new companies?'"

    user_question = st.text_input("Your question:")

    if user_question:
        response = get_bot_response(user_question, change_log_df)
        st.info(f"**Bot:** {response}")

else:
    st.warning("Data could not be loaded. The dashboard is inactive.")

Overwriting app.py


In [15]:
!streamlit run app.py &>/dev/null&

!pip install pyngrok -q
from pyngrok import ngrok

# Terminate open tunnels if any
ngrok.kill()

# --- PASTE YOUR NGROK AUTHTOKEN INSIDE THE QUOTES ---
NGROK_AUTH_TOKEN = "34CtjZfcLuJ6jJ4YYNRPePvR8Kn_5h8D8J8BB8QDyp5isGtnL"
# ----------------------------------------------------

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a tunnel to the streamlit port 8501
public_url = ngrok.connect(8501)
print(f"Click to view your Streamlit app: {public_url}")

Click to view your Streamlit app: NgrokTunnel: "https://pretenseless-reanna-billowiest.ngrok-free.dev" -> "http://localhost:8501"
